In [1]:
%%capture
pip install transformer_lens transformers jaxtyping

In [2]:
import torch
import functools
#import einops
import numpy as np
#import pandas as pd  

#from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int

/opt/homebrew/Caskroom/miniconda/base/envs/algo-neutrality/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def getDevice():
    if torch.cuda.is_available(): #nvidia/runpod
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps") #apple silicon
    else:
        return torch.device("cpu") #not recommended
    
DEVICE = getDevice()
DEVICE

device(type='mps')

In [ ]:
#huggingface authentication
!hf auth login --token HF_TOKEN #replace HF_TOKEN with the actual hf token

In [4]:
#list of models - each model has two different sizes (small ~2B, medium ~8B)
model_list = ['Qwen/Qwen1.5-1.8B-Chat', 'meta-llama/Llama-3.1-8B', 'meta-llama/Llama-3.2-3B', 'gpt2', 'pythia-2.8b-v0', 'qwen2.5-3b', 'qwen3-8b', 'gemma-2-2b', 'gemma-2-7b']

In [23]:
def get_model(model_name):
    # load model from HF and get all the hidden states
    model = HookedTransformer.from_pretrained(model_name, device = DEVICE, dtype=torch.float16, default_padding_side='left', output_hidden_states=True)
    model.eval() #inference mode - no gradients needed
    model.to(DEVICE) #transfer model to device
    return model

In [39]:
def tokenize_prompts(model: HookedTransformer, prompt: str, verbose=False) -> str: #LOVKUSH
    prompt_message = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    if verbose:
        print(model.tokenizer.apply_chat_template(
            prompt_message,
            tokenize=False,
            add_generation_prompt=True
        ))
    prompt_chat_tokenized = model.tokenizer.apply_chat_template(
        prompt_message, tokenize=True, add_generation_prompt=True)
    prompt_chat_str = model.tokenizer.apply_chat_template(
        prompt_message, tokenize=False, add_generation_prompt=True)
    return prompt_chat_tokenized, prompt_chat_str

In [43]:
def get_residual_stream(prompt, which_tokens, model): #combine methods because of run with cache usage for all layers

    # Create an empty tensor to store the residual stream embeddings for each layer
    # Same concept as a accumulator in a loop, but for tensors
    resids = torch.empty(len(prompt), 0, model.cfg.d_model).to(DEVICE)

    #empty tensor to store the residual stream embeddings for each layer
    resids_pre = torch.tensor([]).to(DEVICE)

    #run the model with cache to get the residual stream embeddings for each layer
    _, cache = model.run_with_cache(prompt)
    
    #loop through each layer
    for i in range(model.cfg.n_layers):

        #get the residual stream embeddings for the current layer
        resids_pre = cache[f"blocks.{i}.hook_resid_pre"] # (batch, seq_len, d_model)

        #check if the shape is correct (no_of_prompts, seq_len, d_model)
        assert resids_pre.shape == (1, len(prompt), model.cfg.d_model), f"Expected shape {(1, len(prompt), model.cfg.d_model)}, but got {resids_pre.shape}"
        
        #if the user wants the first token, last token, or mean of all tokens in the sequence
        if (which_tokens == 'first'):
            resids_pre = resids_pre[:, 0:1, :]
        elif (which_tokens == 'last'):
            resids_pre = resids_pre[:, -1:0, :]
        elif (which_tokens == 'mean'):
            # keepdim=True to keep the dimension of the tensor instead of removing it
            resids_pre = resids_pre.mean(dim=1, keepdim=True)  # mean of all tokens
        
        #shape becomes (no_of_prompts, 1, d_model) because we are taking the first/last/mean of the tokens
        assert resids_pre.shape == (1, 1, model.cfg.d_model), f"Expected shape {(1, 1, model.cfg.d_model)}, but got {resids_pre.shape}"

        #using .detach() to detach the tensor from the computational graph and not track the gradients
        # since we are not using the gradients for anything --> we are just using tensor for calculations
        resids_copy = resids_pre.detach().clone()

        #concatenate the residual stream embeddings for the current layer to the tensor
        resids = torch.cat([resids, resids_copy], dim=1)

        #check if the shape is correct (no_of_prompts, no_of_layers, d_model)
        assert resids.shape == (1, i + 1, model.cfg.d_model), f"Expected shape {(1, i + 1, model.cfg.d_model)}, but got {resids.shape}"

    #take the mean of the residual stream embeddings for each layer
    resids = resids.mean(dim=0)

    #check if the shape is correct (no_of_layers, d_model)
    assert resids.shape == (model.cfg.n_layers, model.cfg.d_model), f"Expected shape {(model.cfg.n_layers, model.cfg.d_model)}, but got {resids.shape}"

    return resids

In [40]:
def calculate_steering_vector(X, Y, model):

    # stacks the residual stream embeddings of each layer on top of each other --> (12, 768)

    #Getting the final tensors for the two datasets and calculating the steering vector
    A_mean = get_residual_stream(tokenize_prompts(model, prompt=X), 'mean', model)
    B_mean = get_residual_stream(tokenize_prompts(model, prompt=Y), 'mean', model)

    steering_vector = A_mean - B_mean

    return steering_vector

In [27]:
current_model = get_model(model_list[0]) #get and set the model in use for further calculations and functions

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loaded pretrained model Qwen/Qwen1.5-1.8B-Chat into HookedTransformer
Moving model to device:  mps


In [28]:
p1 = 'Answer the follwing question in French: Who was the first president of USA?'
p2 = 'Answer the follwing question in English: Who was the first president of USA?'
p3 = 'Answer the following in English: Who was the first Tsar of Russia?'


In [29]:
def steered_generation(model, prompt, pos, coeff, steering_vector, layer, token_length):
    tokens = model.to_tokens(prompt)
    
    def steer_model(value: torch.Tensor, hook: HookPoint) -> torch.Tensor:
        value[:, pos, :] += coeff * steering_vector
        return value

    with model.hooks(fwd_hooks=[(f"blocks.{layer}.hook_resid_pre", steer_model)]): 
        steered_output = model.generate(tokens, max_new_tokens=token_length)
        generation =  model.to_string(steered_output)

    return generation

In [ ]:
def normal_generation(model, prompt, token_length):

    #baseline generation 
    _, stringl = tokenize_prompts(model, prompt)
    tokens = model.to_tokens(stringl)

    output = model.generate(tokens, max_new_tokens=token_length)
    generation = model.to_string(output)

    return generation

In [41]:
def generate_with_steering_vector(prompt, model, pos, coeff, layer, token_length):

    steering_vector = calculate_steering_vector(p1, p2, model)

    temp_tensor = steering_vector#[layer:layer+1]

    output = steered_generation(model, prompt, pos, coeff, temp_tensor, layer, token_length)
    print(output)


In [44]:
generate_with_steering_vector(p3, current_model, -1, 1, 14, 30)

AttributeError: 'tuple' object has no attribute 'shape'

In [45]:
normal_generation(current_model, p1, 30)

100%|██████████| 30/30 [00:01<00:00, 17.96it/s]


['<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nAnswer the follwing question in French: Who was the first president of USA?<|im_end|>\n<|im_start|>assistant\n不断地两个不断地2不断地迈向健康不断地过程 serving不断地观察不停地,不断地用 jamaisni Portuguese和深深地 ElementRef x不断地,轻轻地评论提到有条件的基本']